# 8 · AtomLab: ¿es seguro pulsar el AZ-5 ahora?

En el RBMK, el botón de parada de emergencia (AZ-5) podía **aumentar** la potencia durante unos segundos: las
barras llevaban en la punta un desplazador de grafito que, al entrar, expulsaba agua de la parte baja del
núcleo. Con casi todas las barras fuera y mucho vapor, ese empujón bastó en 1986 para reventar el reactor.

¿Se puede saber, mirando la sala de control, si el AZ-5 sería peligroso **ahora**?

* El simulador lo puede calcular (`/api/v1/simulacion/que-pasaria`): copia la central, pulsa el AZ-5 y deja
  pasar 20 s. Es la «verdad», pero en una central real un cálculo así lleva minutos.
* Aquí entrenamos un **modelo sustituto** (*surrogate*): aprende de miles de simulaciones y responde en
  microsegundos.

Datos: el conjunto `atomlab-az5`. Cada fila es un estado de operación al azar (potencia, barras, xenón,
bombas, prueba de inercia…) con las medidas de la sala de control y lo que pasó al pulsar el AZ-5.

In [ ]:
from labia import datos, guardar
import numpy as np, pandas as pd, matplotlib.pyplot as plt

df = datos("atomlab-az5")
print(len(df), "casos ·", df.az5_danado.sum(), f"con rotura de canales ({100 * df.az5_danado.mean():.1f} %)")
df[["reactor_potencia_mwt", "reactor_orm_barras", "reactor_xenon_rel", "reactor_vacio_pct",
    "circuito_bombas_en_marcha", "az5_pico_rel", "az5_danado"]].describe().round(2)

## Dónde está el peligro

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
for d, color, etiqueta in [(df[df.az5_danado == 0], "tab:blue", "sin rotura"), (df[df.az5_danado == 1], "tab:red", "rotura de canales")]:
    ejes[0].scatter(d.reactor_orm_barras, d.az5_pico_rel, s=8, alpha=.5, color=color, label=etiqueta)
    ejes[1].scatter(d.reactor_orm_barras, d.reactor_vacio_pct, s=8, alpha=.5, color=color, label=etiqueta)
ejes[0].set_yscale("log"); ejes[0].set_ylabel("pico de potencia tras el AZ-5 (× la de antes)")
ejes[1].set_ylabel("vacío en el núcleo (%)")
for ax in ejes:
    ax.axvline(15, color="k", ls="--", lw=1); ax.set_xlabel("ORM (barras equivalentes)"); ax.grid(alpha=.3); ax.legend()
ejes[0].set_title("El AZ-5 multiplica la potencia… con pocas barras dentro");

La línea discontinua es el **reglamento** del RBMK: no operar con menos de 15 barras de ORM (aquella noche
había unas 7).

## El reglamento como detector

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

def informe(nombre, real, aviso):
    print(f"{nombre:32s} precisión {100 * precision_score(real, aviso):5.1f} %   detección {100 * recall_score(real, aviso):5.1f} %")

informe("Reglamento: ORM < 15", df.az5_danado, (df.reactor_orm_barras < 15).astype(int))

Detecta **todas** las roturas, pero la mayoría de sus avisos no acabarían en accidente. No es un defecto: un
reglamento de seguridad es **conservador** a propósito (deja margen para lo que no se sabe).

## El modelo sustituto

Entradas: solo lo que se ve en la sala de control (las mismas claves que `GET /api/v1/medidas`).
Los casos son independientes entre sí, así que esta vez sí se pueden separar al azar.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier

ENTRADAS = ["reactor_potencia_mwt", "reactor_orm_barras", "reactor_xenon_rel", "reactor_vacio_pct",
            "reactor_coef_vacio_pcm_pct", "reactor_reactividad_beta", "circuito_bombas_en_marcha",
            "circuito_caudal_pct", "circuito_subenfriamiento_c", "turbina_prueba_inercia"]
X = df[ENTRADAS].astype(float); y = df.az5_danado
X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

arbol = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_ent, y_ent)
bosque = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_ent, y_ent)
informe("Reglamento (prueba)", y_prueba, (X_prueba.reactor_orm_barras < 15).astype(int))
for nombre, m in [("Árbol (profundidad 3)", arbol), ("Bosque aleatorio", bosque)]:
    p = m.predict_proba(X_prueba)[:, 1]
    informe(nombre, y_prueba, (p >= 0.5).astype(int))
    print(f"{'':32s} AUC {roc_auc_score(y_prueba, p):.3f}")
print(export_text(arbol, feature_names=ENTRADAS))

In [ ]:
pd.Series(bosque.feature_importances_, index=ENTRADAS).sort_values().plot.barh(figsize=(7, 4))
plt.title("¿En qué se fija el bosque?");

> Fíjate en `reactor_coef_vacio_pcm_pct` y `reactor_reactividad_beta`: el **coeficiente de vacío** crece cuando
> quedan pocas barras dentro (el agua ya no compite con tantas barras), y una **reactividad** cerca de 1 β
> significa que la central ya está al borde del crítico inmediato. El modelo ha encontrado la física sin que
> nadie se la explique.

## ¿Cuánto más rápido?

Este simulador es sencillo (unas decenas de milisegundos por caso); un código de cálculo de una central real
tarda minutos u horas. Aun así, el modelo responde cientos de casos de una vez en lo que el simulador hace uno.

In [ ]:
import time, requests
v = requests.post("http://atomlab:8040/api/v1/simulacion/que-pasaria", timeout=30).json()
t0 = time.perf_counter(); bosque.predict_proba(X_prueba); t_mod = 1000 * (time.perf_counter() - t0) / len(X_prueba)
print(f"simulador: {v['ms']:.0f} ms por caso (ahora: pico ×{v['az5_pico_rel']})   ·   "
      f"modelo: {t_mod:.3f} ms por caso ({len(X_prueba)} casos de una vez)")

Publica el bosque. El nodo **ia predecir** lo usará cada segundo con las medidas de la sala de control.

In [ ]:
p = bosque.predict_proba(X_prueba)[:, 1]
bosque.fit(X, y)                                     # para publicar, con todos los casos
guardar(bosque, "atomlab-az5", entradas=ENTRADAS, tipo="clasificacion", salida="az5_danado",
        descripcion="Modelo sustituto: ¿rompería canales el AZ-5 pulsado ahora?",
        metricas={"auc": round(float(roc_auc_score(y_prueba, p)), 3),
                  "deteccion_pct": round(100 * float(recall_score(y_prueba, p >= 0.5)), 1),
                  "precision_pct": round(100 * float(precision_score(y_prueba, p >= 0.5)), 1)})

## ✏️ Tu turno

1. Con el umbral 0,5 el bosque no detecta algunas roturas. ¿Qué umbral usarías en una central nuclear? ¿Por qué
   no el mismo que en la Fundición?
2. Entrena una **regresión** de `np.log10(az5_pico_rel)` (cuántas veces se multiplica la potencia). ¿Dónde se
   equivoca más?
3. Quita `reactor_reactividad_beta` y `reactor_coef_vacio_pcm_pct`. ¿Cuánto empeora? ¿Qué variables «de verdad»
   de la central sustituyen a esas dos?
4. Todos los casos tienen las protecciones anuladas y el regulador en manual. ¿Qué pasaría si usaras el modelo con
   la central en automático? (pista: **fuera de lo visto**).
5. ¿Dejarías que este modelo **sustituyera** al reglamento del ORM? ¿Y que lo **complementara**? Argumenta.